<a href="https://colab.research.google.com/github/keyvanmahmoudi/LLM_Engineering_Course/blob/main/solutions/Session16_final_project_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۱۶: پروژه پایانی: دستیار هوشمند فروشگاه

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session16_final_project.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

</div>

In [ ]:
!pip -q install openai sentence-transformers hazm
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/llm_utils.py
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/store_tools.py
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/novinkala_assistant.py

from llm_utils import setup_llm, chat

client, MODEL = setup_llm("ollama")      # or: setup_llm("api")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

پیشنهاد یک‌صفحه‌ای پروژه را بنویسید: مسئله، کاربر هدف، داده و منبع آن، معماری اولیه با نام تکنیک‌های هر بخش، و معیار موفقیت.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

پیشنهاد پروژه یک صفحه است. نمونهٔ زیر را الگو قرار دهید:

---

**عنوان:** دستیار پرسش‌وپاسخ آیین‌نامهٔ آموزشی دانشجویان کارشناسی

**مسئله:** دانشجویان پرسش‌های تکراری دربارهٔ حذف و اضافه، مشروطی و مرخصی تحصیلی دارند و پاسخ آن در آیین‌نامه‌ای ۶۰ صفحه‌ای است. کارشناس آموزش روزانه ده‌ها بار به همین پرسش‌ها جواب می‌دهد.

**کاربر هدف:** دانشجویان کارشناسی؛ کارشناس آموزش برای پرسش‌های پیچیده.

**داده:** فایل Word آیین‌نامه (عمومی)، ۴۰ پرسش واقعی از کانال تلگرام دانشکده با پاسخ کارشناس.

**معماری اولیه:** RAG با تقسیم بر اساس ماده‌های آیین‌نامه و شمارهٔ ماده در متادیتا (جلسه‌های ۷ و ۸)، جستجوی ترکیبی (جلسهٔ ۹)، ارجاع به شمارهٔ ماده و امتناع برای پرسش‌های خارج از آیین‌نامه، رابط Gradio.

**معیار موفقیت:** Hit@3 بیش از ۸۵٪ روی ۴۰ پرسش، درستی پاسخ با نمرهٔ ۴ یا ۵ در بیش از ۷۵٪ پرسش‌ها، امتناع درست برای ۱۰ پرسش خارج از دامنه.

**ریسک‌ها:** آیین‌نامه ممکن است اصلاحیه داشته باشد (نسخهٔ سند در متادیتا)؛ پاسخ اشتباه می‌تواند به تصمیم آموزشی غلط برسد (هشدار و ارجاع به کارشناس در هر پاسخ).

---

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

مجموعهٔ ارزیابی ۳۰ نمونه‌ای پروژه را بسازید، شامل موارد ساده، دشوار و خارج از دامنه، و نسخهٔ اولیه را روی آن ارزیابی کنید.

</div>

In [ ]:
import pandas as pd

# A template for a project evaluation set: one row per test case.
eval_template = pd.DataFrame([
    {"id": "Q01", "type": "simple", "input": "حداکثر تعداد واحد در ترم چند است؟",
     "expected": "۲۰ واحد؛ برای معدل ۱۷ به بالا ۲۴ واحد", "evidence": "ماده ۱۲", "notes": ""},
    {"id": "Q02", "type": "hard", "input": "اگه این ترم مشروط بشم، ترم بعد چند واحد می‌تونم بگیرم؟",
     "expected": "حداکثر ۱۴ واحد", "evidence": "ماده ۱۹", "notes": "needs two articles"},
    {"id": "Q03", "type": "out_of_scope", "input": "شهریه ترم بعد چقدره؟",
     "expected": "REFUSE", "evidence": "", "notes": "not in the regulations"},
])
eval_template.to_csv("eval_set.csv", index=False)
print(eval_template.type.value_counts().to_dict())
eval_template

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

ترکیب پیشنهادی ۳۰ نمونه: حدود ۱۵ پرسش ساده، ۱۰ پرسش دشوار (ترکیبی، محاوره‌ای، با پیش‌فرض غلط) و ۵ پرسش خارج از دامنه. ستون `evidence` برای ارزیابی بازیابی (جلسهٔ ۹) و ستون `type` برای گزارش نتیجه به تفکیک نوع پرسش است. مقادیر بالا فقط نمونه‌اند. پاسخ مرجع را از سند واقعی خودتان بنویسید و ترجیحاً یک نفر دیگر آن را بازبینی کند.

برای ارزیابی نسخهٔ اولیه، از توابع جلسه‌های ۵ و ۹ استفاده کنید: Hit@k برای بازیابی، LLM-as-a-Judge برای درستی پاسخ و بررسی جملهٔ امتناع برای پرسش‌های خارج از دامنه.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

یک بهبود را پیاده کنید و اثرش را با بازهٔ اطمینان گزارش کنید. نمونه‌ها: جستجوی ترکیبی، Reranker، پرامپت بهتر، فاین‌تیون یا مسیریاب.

</div>

In [ ]:
import numpy as np


def paired_bootstrap(correct_a, correct_b, n_boot=2000, seed=0):
    """Difference in accuracy (B - A) with a 95% CI, resampling the same test cases for both systems."""
    a, b = np.asarray(correct_a, float), np.asarray(correct_b, float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(a), size=(n_boot, len(a)))
    diffs = b[idx].mean(axis=1) - a[idx].mean(axis=1)
    return b.mean() - a.mean(), np.percentile(diffs, [2.5, 97.5])


# Example with made-up results for 30 questions: baseline (dense retrieval) vs improvement (hybrid + reranker).
baseline = [1] * 21 + [0] * 9
improved = [1] * 21 + [1] * 5 + [0] * 4
diff, (low, high) = paired_bootstrap(baseline, improved)
print(f"baseline {np.mean(baseline):.0%} -> improved {np.mean(improved):.0%}")
print(f"difference {diff:+.0%}, 95% CI [{low:+.0%}, {high:+.0%}]")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**Bootstrap جفتی** برای هر نمونه‌گیری، **همان** پرسش‌ها را برای هر دو سیستم برمی‌دارد. این روش از Bootstrap جداگانهٔ جلسهٔ ۵ دقیق‌تر است، چون تفاوت دشواری پرسش‌ها را حذف می‌کند. اگر بازهٔ اطمینان اختلاف شامل صفر نباشد، می‌توان گفت بهبود با این داده معنادار است.

در گزارش پروژه این‌ها را بنویسید: معیار، مقدار مبنا، مقدار بعد از بهبود، اختلاف با بازهٔ اطمینان، تعداد نمونه، و یکی دو مثال از پرسش‌هایی که درست شده‌اند و پرسش‌هایی که **خراب** شده‌اند.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

رابط Gradio، README کامل و اسلایدهای ارائهٔ ۱۰ دقیقه‌ای را آماده کنید و چک‌لیست بخش ۵ را تکمیل کنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

یک الگوی README برای پروژه:

```markdown
# دستیار آیین‌نامهٔ آموزشی

دستیاری که به پرسش‌های دانشجویان دربارهٔ آیین‌نامهٔ آموزشی کارشناسی با ارجاع به شمارهٔ ماده پاسخ می‌دهد.

![demo](demo.gif)

## معماری
پرسش ← نرمال‌سازی ← جستجوی ترکیبی (E5 + BM25) ← Reranker ← Qwen2.5-7B با پرامپت مستند به منابع

## اجرا
pip install -r requirements.txt
cp .env.example .env    # LLM_BASE_URL, LLM_API_KEY, LLM_MODEL
python app.py           # Gradio

## ارزیابی (۴۰ پرسش واقعی + ۱۰ خارج از دامنه)
| سیستم | Hit@3 | درستی (۴ یا ۵ از ۵) | امتناع درست |
|---|---|---|---|
| مبنا: جستجوی چگال | ۷۸٪ | ۶۵٪ | ۷/۱۰ |
| ترکیبی + Reranker | ۹۰٪ | ۷۸٪ | ۹/۱۰ |

## محدودیت‌ها
- فقط آیین‌نامهٔ ۱۴۰۲؛ اصلاحیه‌های بعدی وارد نشده‌اند.
- پاسخ‌ها جایگزین نظر کارشناس آموزش نیستند.

## ساختار
assistant/   retrieval.py, generation.py, prompts.py
data/        regulations.docx, eval_set.csv
tests/       test_retrieval.py
app.py       Gradio interface
```

اعداد جدول بالا فقط نمونه‌اند. در README خودتان فقط اعدادی را بنویسید که با اجرای کد قابل بازتولید باشند. برای ارائه، این ترتیب ۱۰ دقیقه‌ای پیشنهاد می‌شود: مسئله و کاربر (۱ دقیقه)، معماری با یک نمودار (۲ دقیقه)، نمایش زنده شامل یک پرسش ساده، یک دشوار و یک خارج از دامنه (۳ دقیقه)، نتایج ارزیابی و یک مثال شکست (۲ دقیقه)، محدودیت‌ها و گام بعدی (۱ دقیقه)، پرسش و پاسخ (۱ دقیقه).

</div>